# Kaggle Predict F1 Pit Stops: EDA and Circuit Context

This notebook builds the evidence base for the modeling workflow. It checks whether the data is reliable, identifies the strongest raw and interaction signals, and records the train/test distribution context that later notebooks need for validation and feature design.

## 1. Setup and Configuration

This section imports the analysis stack, standardizes display and plotting defaults, and defines shared constants such as the target and ID columns. Keeping these values together makes the rest of the notebook easier to audit and keeps metric/plot behavior consistent.

In [ ]:
import gc
import time
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 120)
pd.set_option("display.max_rows", 120)

sns.set_theme(style="whitegrid", palette="viridis")
plt.rcParams.update(
    {
        "figure.dpi": 110,
        "axes.titlesize": 12,
        "axes.labelsize": 10,
        "legend.frameon": False,
    }
)

RANDOM_STATE = 42
TARGET = "PitNextLap"
ID_COL = "id"


## 2. Load Data

Here we read the Kaggle competition files from the fixed competition path, load train/test/submission data, and reduce dataframe memory usage. The goal is to preserve the raw columns needed for EDA and later feature engineering while matching the Kaggle training environment.

In [ ]:
def reduce_memory_usage(df: pd.DataFrame) -> pd.DataFrame:
    """Downcast dataframe columns to reduce notebook memory usage."""
    out = df.copy()
    for col in out.columns:
        dtype = out[col].dtype
        if pd.api.types.is_integer_dtype(dtype):
            out[col] = pd.to_numeric(out[col], downcast="integer")
        elif pd.api.types.is_float_dtype(dtype):
            out[col] = pd.to_numeric(out[col], downcast="float")
        elif pd.api.types.is_object_dtype(dtype):
            nunique = out[col].nunique(dropna=False)
            if nunique / max(len(out), 1) < 0.5:
                out[col] = out[col].astype("category")
    return out


DATA_DIR = Path("/kaggle/input/competitions/playground-series-s6e5")
OUTPUT_DIR = Path("/kaggle/working")

train = reduce_memory_usage(pd.read_csv(DATA_DIR / "train.csv"))
test = reduce_memory_usage(pd.read_csv(DATA_DIR / "test.csv"))
sample_submission = pd.read_csv(DATA_DIR / "sample_submission.csv")

print(f"DATA_DIR: {DATA_DIR}")
print(f"train: {train.shape}")
print(f"test: {test.shape}")
print(f"target positive rate: {train[TARGET].mean():.5f}")
train.head()


## 3. Data Quality

Before interpreting any signal, we check schema shape, data types, missing values, duplicated rows, duplicated IDs, and train/test ID overlap. These checks tell us whether modeling effort should start with cleaning or can move directly into feature and validation design.

In [ ]:
def dataframe_overview(df: pd.DataFrame, name: str) -> pd.DataFrame:
    """Summarize column types, missingness, and unique values."""
    return pd.DataFrame({
        "dataset": name,
        "dtype": df.dtypes.astype(str),
        "missing": df.isna().sum(),
        "missing_pct": df.isna().mean().mul(100),
        "unique": df.nunique(dropna=False),
        "memory_mb": df.memory_usage(deep=True).div(1024 ** 2),
    }).sort_values(["missing_pct", "unique"], ascending=[False, False])

quality_checks = pd.DataFrame({
    "check": [
        "train duplicated rows",
        "test duplicated rows",
        "train duplicated id",
        "test duplicated id",
        "train/test id overlap",
    ],
    "value": [
        int(train.duplicated().sum()),
        int(test.duplicated().sum()),
        int(train[ID_COL].duplicated().sum()),
        int(test[ID_COL].duplicated().sum()),
        int(pd.Index(train[ID_COL]).intersection(test[ID_COL]).shape[0]),
    ],
})

display(pd.concat([
    dataframe_overview(train, "train"),
    dataframe_overview(test, "test"),
]))
display(quality_checks)
train.describe(include="all").T


## 4. Target and Categorical Signal

This section measures the class balance and compares pit-next-lap rates across categorical fields. The target rate sets expectations for average precision, while category-level differences reveal whether strategy identifiers such as compound, race, team, or driver carry meaningful signal.

In [ ]:
target_counts = train[TARGET].value_counts().sort_index()
display(pd.DataFrame({"count": target_counts, "pct": target_counts / len(train)}))
print(f"Positive rate: {train[TARGET].mean():.5f}")

fig, ax = plt.subplots(figsize=(5, 4))
sns.countplot(data=train, x=TARGET, color=sns.color_palette("viridis", 8)[4], ax=ax)
ax.set_title("Target distribution")
plt.show()

categorical_cols = (
    train.drop(columns=[TARGET])
    .select_dtypes(include=["object", "category"])
    .columns.tolist()
)
cat_rows = []
for col in categorical_cols:
    stats = train.groupby(col, observed=True)[TARGET].agg(["count", "mean"]).reset_index()
    stats["feature"] = col
    stats = stats.rename(columns={col: "value", "mean": "target_rate"})
    stats["global_rate_diff"] = stats["target_rate"] - train[TARGET].mean()
    cat_rows.append(stats[["feature", "value", "count", "target_rate", "global_rate_diff"]])
cat_summary = pd.concat(cat_rows, ignore_index=True)
cat_summary.sort_values(["feature", "count"], ascending=[True, False]).head(50)


## 5. Numerical Signal and Outliers

Numerical summaries show the scale, spread, and extreme values of lap, tyre, position, and timing features. We inspect these fields because pit-stop decisions are driven by nonlinear thresholds, and unusually large timing or degradation values can be informative rather than simple noise.

In [ ]:
numeric_cols = [c for c in train.columns if c not in categorical_cols + [TARGET, ID_COL]]
display(train[numeric_cols + [TARGET]].agg(["count", "mean", "std", "min", "median", "max"]).T)

n_cols = 3
n_rows = int(np.ceil(len(numeric_cols) / n_cols))
fig, axes = plt.subplots(n_rows, n_cols, figsize=(16, max(4, 3 * n_rows)))
axes = np.ravel(axes)
for ax, col in zip(axes, numeric_cols):
    sns.histplot(train[col], bins=60, color=sns.color_palette("viridis", 8)[4], ax=ax)
    ax.set_title(col)
for ax in axes[len(numeric_cols):]:
    ax.axis("off")
plt.tight_layout()
plt.show()

outlier_rows = []
for col in numeric_cols:
    low, high = train[col].quantile([0.001, 0.999])
    outlier_rows.append({
        "feature": col,
        "min": train[col].min(),
        "q0.001": low,
        "median": train[col].median(),
        "q0.999": high,
        "max": train[col].max(),
        "below_low": int((train[col] < low).sum()),
        "above_high": int((train[col] > high).sum()),
    })
pd.DataFrame(outlier_rows).sort_values("above_high", ascending=False)


## 6. Strategy Interactions

Pit strategy is rarely explained by one column in isolation. This section studies compound-by-stint behavior and tyre-life-by-race-progress patterns to reveal interactions that should be represented in feature engineering and handled by nonlinear models.

In [ ]:
compound_stint = (
    train.groupby(["Compound", "Stint"], observed=True)[TARGET]
    .agg(["count", "mean"])
    .reset_index()
)
compound_stint_pivot = compound_stint.pivot(
    index="Compound",
    columns="Stint",
    values="mean",
)
fig, ax = plt.subplots(figsize=(10, 4))
sns.heatmap(
    compound_stint_pivot,
    annot=True,
    fmt=".2f",
    cmap="viridis",
    linewidths=0.5,
    cbar_kws={"label": "PitNextLap rate"},
    ax=ax,
)
ax.set_title("PitNextLap rate by compound and stint")
plt.show()

interaction_df = train[["TyreLife", "RaceProgress", TARGET]].copy()
interaction_df["TyreLife_bin"] = pd.qcut(
    interaction_df["TyreLife"],
    q=10,
    duplicates="drop",
)
interaction_df["RaceProgress_bin"] = pd.qcut(
    interaction_df["RaceProgress"],
    q=10,
    duplicates="drop",
)
interaction_rate = interaction_df.pivot_table(
    index="TyreLife_bin",
    columns="RaceProgress_bin",
    values=TARGET,
    aggfunc="mean",
    observed=True,
)
fig, ax = plt.subplots(figsize=(12, 7))
sns.heatmap(
    interaction_rate,
    cmap="viridis",
    cbar_kws={"label": "PitNextLap rate"},
    ax=ax,
)
ax.set_title("PitNextLap rate by tyre life and race progress")
plt.show()


## 7. Train/Test Drift

This section compares train and test distributions using category coverage and numeric PSI. The purpose is to understand whether validation can rely on ordinary stratified folds or whether the test set appears shifted enough to require a more specialized split strategy.

In [ ]:
def psi(
    train_s: pd.Series,
    test_s: pd.Series,
    bins: int = 20,
    eps: float = 1e-6,
) -> float:
    """Calculate population stability index between train and test."""
    edges = np.unique(train_s.quantile(np.linspace(0, 1, bins + 1)).to_numpy())
    if len(edges) <= 2:
        edges = np.linspace(
            min(train_s.min(), test_s.min()),
            max(train_s.max(), test_s.max()),
            bins + 1,
        )
    edges[0], edges[-1] = -np.inf, np.inf
    train_pct = (
        pd.cut(train_s, edges, include_lowest=True)
        .value_counts(normalize=True, sort=False)
        .to_numpy()
    )
    test_pct = (
        pd.cut(test_s, edges, include_lowest=True)
        .value_counts(normalize=True, sort=False)
        .to_numpy()
    )
    train_pct = np.clip(train_pct, eps, None)
    test_pct = np.clip(test_pct, eps, None)
    return float(np.sum((test_pct - train_pct) * np.log(test_pct / train_pct)))

category_coverage = []
for col in categorical_cols:
    train_values = set(train[col].dropna().astype(str))
    test_values = set(test[col].dropna().astype(str))
    category_coverage.append({
        "feature": col,
        "train_unique": len(train_values),
        "test_unique": len(test_values),
        "test_only": len(test_values - train_values),
        "train_only": len(train_values - test_values),
    })
display(pd.DataFrame(category_coverage))

drift = pd.DataFrame([{
    "feature": col,
    "train_mean": train[col].mean(),
    "test_mean": test[col].mean(),
    "mean_diff": test[col].mean() - train[col].mean(),
    "psi": psi(train[col], test[col]),
} for col in numeric_cols]).sort_values("psi", ascending=False)
drift


## 8. Circuit Context

Race-level summaries capture how pit behavior changes by season and Grand Prix. This matters because the same tyre-life value can imply different risk depending on circuit, race length, and strategy pattern.

In [ ]:
race_summary = (
    train.groupby(["Year", "Race"], observed=True)
    .agg(
        rows=(TARGET, "size"),
        pit_next_lap_rate=(TARGET, "mean"),
        mean_lap=("LapNumber", "mean"),
        median_tyre_life=("TyreLife", "median"),
        mean_race_progress=("RaceProgress", "mean"),
    )
    .reset_index()
    .sort_values("pit_next_lap_rate", ascending=False)
)
display(race_summary.head(20))

fig, ax = plt.subplots(figsize=(10, 5))
plot_df = race_summary.head(12).copy()
plot_df["label"] = plot_df["Year"].astype(str) + " " + plot_df["Race"].astype(str)
sns.barplot(
    data=plot_df,
    y="label",
    x="pit_next_lap_rate",
    color=sns.color_palette("viridis", 8)[4],
    ax=ax,
)
ax.set_title("Highest race-level PitNextLap rates")
ax.set_ylabel("")
plt.show()

try:
    import fastf1
    print("FastF1 available for optional circuit-map context.")
except Exception as exc:
    print("FastF1 unavailable; circuit-map drawing skipped.")
    print(exc)


## 9. EDA Summary

The EDA supports a clean modeling path: the dataset is stable, the target rate is about 20%, compound/stint and tyre-life/race-progress interactions are central, lap-time and degradation extremes are useful, and train/test drift is low. The next notebooks should prioritize tree models, compact row-safe ratio features, and race-level calibration diagnostics.